# ESOREX quickstart: predicting an enzyme's substrate preferences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UCB-BioE-Anderson-Lab/ESOREX/blob/main/notebooks/tyrb_quickstart.ipynb)

Enzymes accept some molecules and reject others. **From only a handful of measured substrates,**
**can we predict how fast an enzyme will turn over molecules it has never seen?**

ESOREX trains on the 9 natural amino-acid substrates of *E. coli* **TyrB** (an aromatic-preferring
aminotransferase) and predicts 14 *unnatural* analogs spanning six orders of magnitude in rate.
The last cell is the payoff: predicted vs measured for all 14. Runs in about a minute.

## 1. Install and fetch the data

Only RDKit needs installing (NumPy, SciPy, pandas ship with Colab). We clone the repo for the
`esorex` package and the TyrB dataset.

In [ ]:
!pip install -q rdkit          # numpy, scipy and pandas are preinstalled in Colab (do not downgrade them)
!git clone -q https://github.com/UCB-BioE-Anderson-Lab/ESOREX.git
import sys
sys.path.insert(0, "/content/ESOREX")
DATA = "/content/ESOREX/data/transaminases/ONUFFER_curated.csv"

## 2. Give the model only the 9 naturals

TyrB prefers aromatic side chains. We hand it just the 9 amino acids it evolved with; every other
measured substrate is held out as an unseen test. The **reactive core** (amine, alpha-carbon,
carboxyl) is matched with SMARTS; specificity is learned from everything outside it.

In [ ]:
import csv
from rdkit import Chem

RATE_COL  = "eTATase_kf_over_KD_M-1_s-1"
DUPLICATE = {"Arginine (mu = 1.0)"}
NATURALS  = {"Aspartate", "Glutamate", "Phenylalanine", "Tryptophan", "Tyrosine",
             "Alanine", "Leucine", "Valine", "Arginine (mu = 0.2)"}
AA_CORE   = Chem.MolFromSmarts("[NX3][CX4H][CX3](=O)[OX2]")

def core(mol):
    return set(mol.GetSubstructMatch(AA_CORE))

def parse_rate(s):
    s = s.strip()
    if not s or s.lower() in ("nd", "n/a", "na", "-"): return None
    try: return float(s)
    except ValueError: return None

naturals, analogs = [], []
with open(DATA) as f:
    for row in csv.DictReader(f):
        name = row["substrate"].strip()
        if name in DUPLICATE: continue
        rate = parse_rate(row[RATE_COL])
        mol  = Chem.MolFromSmiles(row.get("substrate_smiles", "").strip())
        if rate is None or mol is None or not core(mol): continue
        (naturals if name in NATURALS else analogs).append((name, mol, rate))

print(f"trained on {len(naturals)} naturals; predicting {len(analogs)} unseen analogs")

## 3. Train, reproducing every measurement exactly

ESOREX converts rates to activation energies (`E = -RT ln k`) and solves the free-energy
decomposition as a **hard constraint**. `exact fit: True` means it reproduces all 9 training
rates exactly, interpolation, not a lossy regression.

In [ ]:
from esorex.energetic_specificity import EnergeticSpecificityModel

model = EnergeticSpecificityModel()
info = model.train([m for _, m, _ in naturals],
                   [core(m) for _, m, _ in naturals],
                   rates=[r for _, _, r in naturals])
print("exact fit:", info["exact"])

## 4. Predict the 14 unseen analogs

One call per molecule. `fold_error` is how far off the prediction is, as a multiple of the
measured rate: `1.0` is exact, `2.0` is a factor of two either way.

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

rows = []
for name, mol, measured in analogs:
    p = model.predict(mol, core(mol))
    rows.append(dict(substrate=name, measured_rate=measured, predicted_rate=round(p.rate, 1)))

df = pd.DataFrame(rows)
df["fold_error"] = (df[["measured_rate", "predicted_rate"]].max(axis=1) /
                    df[["measured_rate", "predicted_rate"]].min(axis=1)).round(1)
df = df.sort_values("measured_rate", ascending=False).reset_index(drop=True)

rho = spearmanr(df.measured_rate, df.predicted_rate).correlation
print(f"held-out rank correlation (Spearman rho) = {rho:.2f}")
print(f"median fold error = {df.fold_error.median():.1f}x   |   "
      f"within 10x: {(df.fold_error <= 10).sum()} of {len(df)}")
df

## 5. The payoff: predicted vs measured

All 14 unseen analogs, on log axes spanning six orders of magnitude. The dashed line is a perfect
prediction and the shaded band is within 10x. Most analogs land in the band; the misses are the
ones whose chemistry the 9 naturals never covered.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

logerr = (np.log10(df.predicted_rate) - np.log10(df.measured_rate)).abs()
fig, ax = plt.subplots(figsize=(6.8, 5.6))
lims = [min(df.measured_rate.min(), df.predicted_rate.min()) * 0.3,
        max(df.measured_rate.max(), df.predicted_rate.max()) * 3]
band = np.array(lims)
ax.fill_between(band, band / 10, band * 10, color="#4c72b0", alpha=0.10,
                zorder=0, label="within 10x")
ax.plot(lims, lims, "--", color="#9aa0a6", zorder=1, label="perfect prediction")
ax.scatter(df.measured_rate, df.predicted_rate, s=85, color="#4c72b0",
           edgecolor="#1b2a41", linewidth=0.6, zorder=3)
(ax.set_xscale("log"), ax.set_yscale("log"))
ax.set_xlim(lims); ax.set_ylim(lims)
(ax.set_xlabel("measured rate  (kf/KD, M$^{-1}$s$^{-1}$)"), ax.set_ylabel("predicted rate"))
ax.set_title(f"TyrB: 9 trained, {len(df)} predicted unseen\n"
             f"rank accuracy rho = {rho:.2f}   |   median fold error = {df.fold_error.median():.0f}x",
             fontsize=11)
w = df.loc[logerr.idxmax()]
ax.annotate(f"{w.substrate}: ~{w.fold_error:.0f}x low\n(long aliphatic chain, no aromatic ring)",
            xy=(w.measured_rate, w.predicted_rate), xycoords="data",
            xytext=(0.45, 0.09), textcoords="axes fraction", fontsize=9, color="#8a2b0e",
            arrowprops=dict(arrowstyle="->", color="#8a2b0e", lw=0.9))
ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
plt.tight_layout(); plt.show()

### The point

From just **9 measurements**, ESOREX ranks 14 substrates it never saw (Spearman rho about 0.73)
and puts 9 of them within 10x of the measured rate, median error about 4x, across a range that
spans six orders of magnitude.

The failure mode is the interesting part, and it is chemical, not statistical. TyrB's naturals are
aromatics plus short aliphatics, so the model has never priced a long, flexible alkyl chain: its
worst miss is **2-aminooctanoate** (~900x low), followed by norleucine and cyclohexylalanine. Add
one such substrate to the training set and that whole family comes into range.

Next: the same model does regioselectivity (FucTIII) in `docs/demonstrations/`; the concept pages
are in `docs/`. Point ESOREX at your own atom-mapped reactions and rates to model a different enzyme.